# Run `mapFolding` meanders on Colab

This notebook is a temporary Colab workaround for a Python 3.12 runtime. It clones `mapFolding` from GitHub, installs only the dependencies needed for the meanders workflow, and then runs the meanders cell below.

## What it does

1. Clone `mapFolding` from GitHub.

2. Install the meanders dependencies needed on Colab Python 3.12.

3. Add the clone to `sys.path`.

4. Run the meanders cell below, which is virtually identical to `easyRun/meanders.py`.



In [ ]:
from __future__ import annotations
from pathlib import Path
import subprocess, sys

if 'google.colab' in sys.modules:
	from google.colab import drive, userdata  # pyright: ignore[reportUnusedImport, reportUnknownVariableType, reportMissingImports] # ty: ignore[unresolved-import]
	drive.mount('/content/drive')  # pyright: ignore[reportUnknownMemberType]

url = 'https://github.com/hunterhogan/mapFolding.git'
repoPath: Path = Path.cwd() / 'mapFolding'
overwriteClone = False

def run(command: list[str]) -> None:
	print('>>', ' '.join(command))
	subprocess.run(command, check=True)

print('Python', sys.version)
print('Working directory:', Path.cwd())
listPackages: list[str] = [
	'gmpy2'
	, 'hunterMakesPy>=0.7.3'
	, 'numpy'
	, 'pandas'
	, 'platformdirs'
	, 'tqdm'
	, 'urllib3'
]
if repoPath.exists() and overwriteClone:
	run(['git', '-C', str(repoPath), 'fetch', '--all', '--prune'])
	run(['git', '-C', str(repoPath), 'reset', '--hard', 'origin/main'])
	run(['git', '-C', str(repoPath), 'clean', '-fdx'])
elif not repoPath.exists():
	run(['git', 'clone', url, str(repoPath)])

%pip install {' '.join(listPackages)}
if str(repoPath) not in sys.path:
	sys.path.insert(0, str(repoPath))
print('Clone/install complete.')
print('Repo path:', repoPath)

# Run

In [ ]:
from mapFolding.basecamp import countMeanders
from mapFolding.oeis import printEasyRunBenchmark, printEasyRunHeader
from pathlib import Path
from typing import TYPE_CHECKING
import gc, sys, time, warnings

if TYPE_CHECKING:
	from os import PathLike

if __name__ == '__main__':
	if (3, 14) <= sys.version_info:
		warnings.filterwarnings("ignore", category=FutureWarning)

	pathLikeWrite: PathLike[str] | None = Path('/apps/mapFolding/mapFolding/jobs')
	pathLikeWrite = None
	flow = 'matrixMeanders'
	flow = 'matrixPandas'
	flow = 'matrixNumPy'

	for oeisID, kind in [
			('A005316', 'meanders'),
			# ('A000682', 'semi'),
		]:
		printEasyRunHeader(oeisID, flow)

		"""# Identifiers. improve
		"generate up to four targets."
		1. Adding a new loop.
		2. Dragging up a loop end.
		3. Dragging down a loop end.
		4. Connect ends across the line.
		"""

		nList: list[int] = []
		nList.extend(range(2, 10))
		nList.extend(range(10, 28))
		# nList.extend(range(28, 33))
		# nList.extend(range(33, 38))
		# nList.extend(range(38, 43))
		# nList.extend(range(43, 45))
		# nList.extend(range(45, 50))

		for n in nList:
			gc.collect()
			timeStart: float = time.perf_counter()
			countTotal: int = countMeanders(kind, n, flow, pathLikeWrite)  # pyright: ignore[reportArgumentType] # ty: ignore[invalid-argument-type]

			printEasyRunBenchmark(oeisID, n, countTotal, timeStart, ratio=False)
